In [1]:
# Imports for a simple, reproducible Part 2 baseline and model.
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, log_loss

PHASES = ["I", "II", "III"]
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]


In [2]:
# Find the shared project data even if Jupyter starts from another folder.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [path / "applied-ml-midterm" for path in candidates.copy()]

PROJECT_DIR = next((path for path in candidates if (path / "part2-data" / "outcome_train.csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part2-data from {cwd}")
DATA_DIR = PROJECT_DIR / "part2-data"
OUTPUT_DIR = PROJECT_DIR / "deliverables" / "reproduced" / "part2_supplementary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outcome_train = pd.read_csv(DATA_DIR / "outcome_train.csv")
outcome_val = pd.read_csv(DATA_DIR / "outcome_val.csv")
outcome_test = pd.read_csv(DATA_DIR / "outcome_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "part2_sample_submission.csv")
assert outcome_train.ID.is_unique and outcome_val.ID.is_unique and outcome_test.ID.is_unique
assert outcome_test.ID.equals(sample_submission.ID)
assert set(outcome_train.trial_id).isdisjoint(outcome_val.trial_id)
assert set(outcome_test.trial_id).isdisjoint(outcome_train.trial_id)
assert set(outcome_test.trial_id).isdisjoint(outcome_val.trial_id)
print("Train / validation / test:", len(outcome_train), len(outcome_val), len(outcome_test))
print("Train phases:", outcome_train.phase.value_counts().reindex(PHASES).to_dict())
print("Validation phases:", outcome_val.phase.value_counts().reindex(PHASES).to_dict())
print("Train success rates:", outcome_train.groupby("phase").label.mean().reindex(PHASES).round(3).to_dict())


Train / validation / test: 8037 1013 1500
Train phases: {'I': 1012, 'II': 3950, 'III': 3075}
Validation phases: {'I': 149, 'II': 501, 'III': 363}
Train success rates: {'I': 0.566, 'II': 0.477, 'III': 0.653}


In [3]:
# Baseline: every trial gets its phase's success rate from training only.
phase_success_rates = outcome_train.groupby("phase").label.mean().to_dict()
assert set(outcome_val.phase).issubset(phase_success_rates)
assert set(outcome_test.phase).issubset(phase_success_rates)
baseline_val_probability = outcome_val.phase.map(phase_success_rates).to_numpy(dtype=float)
baseline_test_probability = outcome_test.phase.map(phase_success_rates).to_numpy(dtype=float)
print("Baseline phase success rates:", {p: round(phase_success_rates[p], 4) for p in PHASES})


Baseline phase success rates: {'I': 0.5662, 'II': 0.477, 'III': 0.6533}


In [4]:
# Improvement: TF-IDF of eligibility criteria plus a one-hot phase indicator.
# Fit both transforms and logistic regression only on outcome_train.
def create_outcome_features(train_df, val_df, test_df):
    tfidf = TfidfVectorizer(min_df=3, max_features=15000,
                            ngram_range=(1, 2), sublinear_tf=True)
    train_text = tfidf.fit_transform(train_df.criteria.fillna(""))
    val_text = tfidf.transform(val_df.criteria.fillna(""))
    test_text = tfidf.transform(test_df.criteria.fillna(""))

    phase_encoder = OneHotEncoder(handle_unknown="ignore")
    train_phase = phase_encoder.fit_transform(train_df[["phase"]])
    val_phase = phase_encoder.transform(val_df[["phase"]])
    test_phase = phase_encoder.transform(test_df[["phase"]])
    return (hstack([train_text, train_phase]).tocsr(),
            hstack([val_text, val_phase]).tocsr(),
            hstack([test_text, test_phase]).tocsr(), tfidf, phase_encoder)

X_train, X_val, X_test, tfidf, phase_encoder = create_outcome_features(
    outcome_train, outcome_val, outcome_test
)
outcome_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                   random_state=42)
outcome_model.fit(X_train, outcome_train.label.to_numpy(dtype=int))
model_val_probability = outcome_model.predict_proba(X_val)[:, 1]
model_test_probability = outcome_model.predict_proba(X_test)[:, 1]
print("Feature matrix:", X_train.shape, "TF-IDF vocabulary:", len(tfidf.vocabulary_))


Feature matrix: (8037, 15003) TF-IDF vocabulary: 15000


In [5]:
# Validation: the competition uses log loss; also report each phase's F1, ROC-AUC, and AP.
def evaluate_outcome_model(df, probabilities, model_name):
    truth = df.label.to_numpy(dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if len(probabilities) != len(df) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid probability vector")
    rows = []
    for phase in [*PHASES, "Overall"]:
        mask = np.ones(len(df), dtype=bool) if phase == "Overall" else df.phase.eq(phase).to_numpy()
        y, p = truth[mask], probabilities[mask]
        rows.append({"Model": model_name, "Phase": phase, "Trials": int(mask.sum()),
                     "F1": f1_score(y, p >= 0.5, zero_division=0),
                     "ROC-AUC": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                     "Average precision": average_precision_score(y, p),
                     "Log loss": log_loss(y, p, labels=[0, 1])})
    return pd.DataFrame(rows)

validation_scores = pd.concat([
    evaluate_outcome_model(outcome_val, baseline_val_probability, "Phase prior"),
    evaluate_outcome_model(outcome_val, model_val_probability, "Criteria + phase LR"),
], ignore_index=True)
print(validation_scores.round(4).to_string(index=False))
baseline_loss = validation_scores.query('Model == "Phase prior" and Phase == "Overall"')["Log loss"].iloc[0]
model_loss = validation_scores.query('Model == "Criteria + phase LR" and Phase == "Overall"')["Log loss"].iloc[0]
selected_model = "Criteria + phase LR" if model_loss < baseline_loss else "Phase prior"
selected_val_probability = model_val_probability if selected_model == "Criteria + phase LR" else baseline_val_probability
selected_test_probability = model_test_probability if selected_model == "Criteria + phase LR" else baseline_test_probability
print("Selected by lower validation log loss:", selected_model)


              Model   Phase  Trials     F1  ROC-AUC  Average precision  Log loss
        Phase prior       I     149 0.7265   0.5000             0.5705    0.6832
        Phase prior      II     501 0.0000   0.5000             0.4711    0.6915
        Phase prior     III     363 0.7960   0.5000             0.6612    0.6404
        Phase prior Overall    1013 0.6058   0.5925             0.6120    0.6720
Criteria + phase LR       I     149 0.7692   0.7820             0.8380    0.5612
Criteria + phase LR      II     501 0.6422   0.7358             0.7165    0.6013
Criteria + phase LR     III     363 0.8300   0.7655             0.8578    0.5403
Criteria + phase LR Overall    1013 0.7477   0.7689             0.8049    0.5736
Selected by lower validation log loss: Criteria + phase LR


In [6]:
# Part 1 reasons enter only this error audit, never the outcome classifier.
# Use visible Part 1 train/validation labels and the team's selected recovered train labels.
def create_reason_lookup(project_dir):
    original_train = pd.read_csv(project_dir / "part1-data" / "part1_train.csv")
    original_val = pd.read_csv(project_dir / "part1-data" / "part1_val.csv")
    original_test = pd.read_csv(project_dir / "part1-data" / "part1_test.csv")
    recovered = pd.read_csv(project_dir / "part1-results" / "knn-experiment" /
                            "best_augmented_training_labels.csv")
    test_predictions = pd.read_csv(project_dir / "part1-results" / "knn-experiment" /
                                   "best_augmented_candidate_submission.csv")
    assert original_train.ID.is_unique and recovered.ID.is_unique and original_val.ID.is_unique
    assert set(original_train.ID) == set(recovered.ID)
    recovered = recovered.set_index("ID").loc[original_train.ID].reset_index()
    observed_mask = original_train[REASONS].isin([0, 1]).all(axis=1)
    hidden_mask = original_train[REASONS].eq(-100).all(axis=1)
    assert (observed_mask | hidden_mask).all()
    assert (recovered.loc[observed_mask, REASONS].to_numpy() ==
            original_train.loc[observed_mask, REASONS].to_numpy()).all()
    assert recovered[REASONS].isin([0, 1]).all().all()
    train_lookup = recovered[["ID", *REASONS]].copy()
    train_lookup["reason_source"] = np.where(observed_mask, "observed", "predicted")
    val_lookup = original_val[["ID", *REASONS]].copy()
    val_lookup["reason_source"] = "observed"
    assert original_test.ID.equals(test_predictions.ID)
    assert test_predictions.Predicted.notna().all()
    test_lookup = test_predictions[["ID"]].copy()
    reason_sets = test_predictions.Predicted.str.split()
    assert reason_sets.map(bool).all()
    assert reason_sets.map(lambda values: set(values).issubset(REASONS)).all()
    for reason in REASONS:
        test_lookup[reason] = reason_sets.map(lambda values: int(reason in values))
    test_lookup["reason_source"] = "predicted"
    lookup = pd.concat([train_lookup, val_lookup, test_lookup], ignore_index=True)
    lookup = lookup.rename(columns={"ID": "trial_id"})
    assert lookup.trial_id.is_unique
    return lookup

reason_lookup = create_reason_lookup(PROJECT_DIR)
unsuccessful = outcome_val.loc[outcome_val.label.eq(0), ["ID", "trial_id", "phase"]].copy()
unsuccessful["Phase prior predicted success"] = baseline_val_probability[outcome_val.label.eq(0).to_numpy()] >= 0.5
unsuccessful["Criteria + phase LR predicted success"] = model_val_probability[outcome_val.label.eq(0).to_numpy()] >= 0.5
unsuccessful = unsuccessful.merge(reason_lookup, on="trial_id", how="left", validate="many_to_one")
unsuccessful["reason_source"] = unsuccessful.reason_source.fillna("no recorded stopping reason")
assert unsuccessful.ID.is_unique

def reason_error_table(failures):
    rows = []
    for phase in PHASES:
        part = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = part.reason_source.eq("no recorded stopping reason") if reason == "No recorded stopping reason" else part[reason].eq(1)
            group = part.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed reasons": int(group.reason_source.eq("observed").sum()),
                   "Predicted reasons": int(group.reason_source.eq("predicted").sum())}
            for name in ["Phase prior", "Criteria + phase LR"]:
                mistakes = int(group[f"{name} predicted success"].sum())
                row[f"{name} false successes"] = mistakes
                row[f"{name} false-success rate"] = mistakes / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

reason_errors = reason_error_table(unsuccessful)
print("Unsuccessful validation rows:", len(unsuccessful))
print("Reason sources:", unsuccessful.reason_source.value_counts().to_dict())
print(reason_errors.round(3).to_string(index=False))


Unsuccessful validation rows: 452
Reason sources: {'no recorded stopping reason': 254, 'predicted': 171, 'observed': 27}
Phase                Reason group  Trials  Observed reasons  Predicted reasons  Phase prior false successes  Phase prior false-success rate  Criteria + phase LR false successes  Criteria + phase LR false-success rate
    I                      Safety       8                 1                  7                            8                             1.0                                    4                                   0.500
    I                     Funding       3                 1                  2                            3                             1.0                                    2                                   0.667
    I                    Efficacy       4                 0                  4                            4                             1.0                                    3                                   0.750
    I      

In [7]:
# Export the selected model's success probability for every test ID.
submission = pd.DataFrame({"ID": outcome_test.ID,
                           "label": selected_test_probability})
if not submission.ID.equals(sample_submission.ID):
    raise ValueError("Submission ID order differs from sample submission")
if (not submission.ID.is_unique or submission.label.isna().any() or
        not np.isfinite(submission.label.to_numpy()).all() or
        not submission.label.between(0, 1).all()):
    raise ValueError("Invalid submission")
submission_path = OUTPUT_DIR / "part2_simple_submission.csv"
submission.to_csv(submission_path, index=False)
assert pd.read_csv(submission_path).shape == (1500, 2)
print("Saved:", submission_path)
print("Rows:", len(submission), "Probability range:",
      (round(submission.label.min(), 4), round(submission.label.max(), 4)))


Saved: [original local path] Machine Learning/midterm/applied-ml-midterm/original_attempts/part_2/part2_simple_submission.csv
Rows: 1500 Probability range: (np.float64(0.1353), np.float64(0.9552))


In [8]:
# Adapted from HW3 part1.ipynb: reusable preprocessing without global counters or downloads.
import ast
import re
import nltk
# Bundled resources allow preprocessing without a network connection.
nltk.data.path.insert(0, str(PROJECT_DIR / "deliverables" / "nltk_data"))
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
english_stopwords = set(stopwords.words("english"))
common_clinical_stopwords = {
    "patient", "subject", "must", "month", "year", "least", "week", "visit",
    "use", "study", "prior", "within", "day", "known", "time", "including",
    "age", "eg", "following", "previous", "greater", "status", "would",
    "allowed", "history", "disease", "kg", "taking", "currently", "able",
    "mg", "g", "per", "therapy", "treatment",
}
criteria_stopwords = english_stopwords | common_clinical_stopwords | {"ii", "iii"}
disease_stopwords = (english_stopwords | common_clinical_stopwords) - {"i"}
drug_stopwords = english_stopwords | common_clinical_stopwords | {"ml", "mcg", "+"}

def _parse_list_items(value):
    """Flatten CSV-encoded lists; handle missing/malformed values as plain text."""
    if value is None or (isinstance(value, (float, np.floating)) and np.isnan(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        return [item for part in value for item in _parse_list_items(part)]
    if not isinstance(value, str):
        return [str(value)]
    value = value.strip()
    if not value or value.lower() in {"nan", "none"}:
        return []
    if value.startswith(("[", "(", "{")):
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [value]
        if isinstance(parsed, (list, tuple, set)):
            return _parse_list_items(parsed)
        return [str(parsed)]
    return [value]

def pre_process_criteria(value):
    """HW3 criteria: lowercase, remove headings, lemmatize, drop stopwords/digits."""
    if not isinstance(value, str) or not value.strip():
        return ""
    text = value.strip().lower().replace("inclusion criteria:", "").replace("exclusion criteria:", "")
    words = []
    for token in re.split(r"[\s\W]+", text):
        token = lemmatizer.lemmatize(token)
        if token and token not in criteria_stopwords and not re.search(r"\d|®", token):
            words.append(token)
    return " ".join(words)

def pre_process_diseases_cleaned(value):
    """HW3 disease list handling, with safe parsing and consistent lowercase."""
    words = []
    for item in _parse_list_items(value):
        for token in re.split(r"[\s/]+", item.lower()):
            token = token.strip("(), ")
            if token and token not in disease_stopwords and token not in {"-", "/"}:
                words.append(token)
    return " ".join(words)

def pre_process_drugs(value):
    """HW3 drug list handling, filtering doses, units and standalone numbers."""
    words = []
    for item in _parse_list_items(value):
        for token in re.split(r"[\s/]+", item.lower()):
            token = token.strip("(), ")
            if (not token or token in drug_stopwords or token in {"-", "/"}
                    or "%" in token or re.fullmatch(r"[\d.]+", token)
                    or re.fullmatch(r"\d+(?:\.\d+)?m(?:c)?g", token)
                    or (token.startswith("[") and token.endswith("]"))):
                continue
            words.append(token)
    return " ".join(words)

def pre_process_icd(value):
    """HW3 nested ICD list handling: remove periods and append `icd`."""
    codes = []
    for item in _parse_list_items(value):
        # Valid nested-list values are parsed above. For malformed strings,
        # recover code-shaped tokens instead of raising an exception.
        for code in re.findall(r"[A-Za-z][0-9]{2}(?:\.[A-Za-z0-9]+)?", item):
            codes.append(code.replace(".", "") + "icd")
    return " ".join(codes)

# These four functions are independent; the controlled model test below uses criteria only.
# Like the original HW3 function, criteria removes negation stopwords (e.g. no/not)
# and every token containing a digit, possibly losing clinical qualifiers, ages or doses.
examples = outcome_train.iloc[0]
print("Criteria before:", str(examples.criteria).strip()[:130].replace("\n", " "))
print("Criteria after:", pre_process_criteria(examples.criteria)[:130])
print("Disease example:", pre_process_diseases_cleaned(examples.diseases_cleaned))
print("Drug example:", pre_process_drugs(examples.drugs))
print("ICD example:", pre_process_icd(examples.icdcodes)[:100])
print("Malformed/empty values:", [pre_process_diseases_cleaned("[broken"),
                                   pre_process_drugs(np.nan), pre_process_icd("[]")])


Criteria before: Inclusion Criteria:            -  Type 2 diabetes mellitus patients            -  Subject is only on a diet and exercise program o
Criteria after: type diabetes mellitus diet exercise program ha received single antihyperglycemic agent two antihyperglycemic agent excluding insu
Disease example: type 2 diabetes mellitus
Drug example: ipragliflozin
ICD example: E1165icd E119icd E1121icd E1136icd E1141icd E1142icd E1144icd
Malformed/empty values: ['[broken', '', '']


In [9]:
# Controlled comparison: change only criteria preprocessing, not TF-IDF/LR settings.
processed_train_criteria = outcome_train.criteria.map(pre_process_criteria)
processed_val_criteria = outcome_val.criteria.map(pre_process_criteria)
processed_test_criteria = outcome_test.criteria.map(pre_process_criteria)

processed_tfidf = TfidfVectorizer(min_df=3, max_features=15000,
                                  ngram_range=(1, 2), sublinear_tf=True)
processed_train_text = processed_tfidf.fit_transform(processed_train_criteria)
processed_val_text = processed_tfidf.transform(processed_val_criteria)
processed_test_text = processed_tfidf.transform(processed_test_criteria)
processed_X_train = hstack([processed_train_text,
                           phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
processed_X_val = hstack([processed_val_text,
                         phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
processed_X_test = hstack([processed_test_text,
                          phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
processed_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                     random_state=42)
processed_model.fit(processed_X_train, outcome_train.label.to_numpy(dtype=int))
processed_val_probability = processed_model.predict_proba(processed_X_val)[:, 1]
processed_test_probability = processed_model.predict_proba(processed_X_test)[:, 1]
processed_scores = evaluate_outcome_model(outcome_val, processed_val_probability,
                                          "HW3 criteria + phase LR")
preprocessing_comparison = pd.concat([validation_scores, processed_scores], ignore_index=True)
print(preprocessing_comparison.round(4).to_string(index=False))
print("Criteria token counts (raw / HW3 processed):",
      round(outcome_train.criteria.fillna("").str.split().map(len).mean(), 1),
      round(processed_train_criteria.str.split().map(len).mean(), 1))
print("Processed TF-IDF vocabulary:", len(processed_tfidf.vocabulary_))
print("No submission changed; this is a separate validation comparison.")


                  Model   Phase  Trials     F1  ROC-AUC  Average precision  Log loss
            Phase prior       I     149 0.7265   0.5000             0.5705    0.6832
            Phase prior      II     501 0.0000   0.5000             0.4711    0.6915
            Phase prior     III     363 0.7960   0.5000             0.6612    0.6404
            Phase prior Overall    1013 0.6058   0.5925             0.6120    0.6720
    Criteria + phase LR       I     149 0.7692   0.7820             0.8380    0.5612
    Criteria + phase LR      II     501 0.6422   0.7358             0.7165    0.6013
    Criteria + phase LR     III     363 0.8300   0.7655             0.8578    0.5403
    Criteria + phase LR Overall    1013 0.7477   0.7689             0.8049    0.5736
HW3 criteria + phase LR       I     149 0.7624   0.7645             0.8211    0.5749
HW3 criteria + phase LR      II     501 0.6522   0.7475             0.7352    0.5938
HW3 criteria + phase LR     III     363 0.8266   0.7753          

In [10]:
# HW3-style combined text: cleaned criteria + diseases + drugs + ICD codes.
# This changes the text fields only. TF-IDF, phase encoding, C and solver stay fixed.
def make_hw3_combined_text(df):
    cleaned = pd.DataFrame({
        "criteria_pp": df.criteria.map(pre_process_criteria),
        "diseases_cleaned_pp": df.diseases_cleaned.map(pre_process_diseases_cleaned),
        "drugs_pp": df.drugs.map(pre_process_drugs),
        "icdcodes_pp": df.icdcodes.map(pre_process_icd),
    })
    return cleaned.astype(str).agg(" ".join, axis=1).str.strip(), cleaned

combined_train_text, combined_train_fields = make_hw3_combined_text(outcome_train)
combined_val_text, combined_val_fields = make_hw3_combined_text(outcome_val)
combined_test_text, combined_test_fields = make_hw3_combined_text(outcome_test)
assert combined_train_fields.criteria_pp.equals(processed_train_criteria)
assert combined_val_fields.criteria_pp.equals(processed_val_criteria)
assert combined_test_fields.criteria_pp.equals(processed_test_criteria)

combined_tfidf = TfidfVectorizer(min_df=3, max_features=15000,
                                 ngram_range=(1, 2), sublinear_tf=True)
combined_train_features = combined_tfidf.fit_transform(combined_train_text)
combined_val_features = combined_tfidf.transform(combined_val_text)
combined_test_features = combined_tfidf.transform(combined_test_text)
combined_X_train = hstack([combined_train_features,
                           phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
combined_X_val = hstack([combined_val_features,
                         phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
combined_X_test = hstack([combined_test_features,
                          phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
combined_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                    random_state=42)
combined_model.fit(combined_X_train, outcome_train.label.to_numpy(dtype=int))
combined_val_probability = combined_model.predict_proba(combined_X_val)[:, 1]
combined_test_probability = combined_model.predict_proba(combined_X_test)[:, 1]
combined_scores = evaluate_outcome_model(outcome_val, combined_val_probability,
                                         "HW3 combined + phase LR")
all_text_scores = pd.concat([validation_scores, processed_scores, combined_scores],
                            ignore_index=True)
print(all_text_scores.round(4).to_string(index=False))
print("Combined example:", combined_train_text.iloc[0][:180])
print("Average token counts, cleaned criteria / combined:",
      round(processed_train_criteria.str.split().map(len).mean(), 1),
      round(combined_train_text.str.split().map(len).mean(), 1))
print("Combined TF-IDF vocabulary:", len(combined_tfidf.vocabulary_))


                  Model   Phase  Trials     F1  ROC-AUC  Average precision  Log loss
            Phase prior       I     149 0.7265   0.5000             0.5705    0.6832
            Phase prior      II     501 0.0000   0.5000             0.4711    0.6915
            Phase prior     III     363 0.7960   0.5000             0.6612    0.6404
            Phase prior Overall    1013 0.6058   0.5925             0.6120    0.6720
    Criteria + phase LR       I     149 0.7692   0.7820             0.8380    0.5612
    Criteria + phase LR      II     501 0.6422   0.7358             0.7165    0.6013
    Criteria + phase LR     III     363 0.8300   0.7655             0.8578    0.5403
    Criteria + phase LR Overall    1013 0.7477   0.7689             0.8049    0.5736
HW3 criteria + phase LR       I     149 0.7624   0.7645             0.8211    0.5749
HW3 criteria + phase LR      II     501 0.6522   0.7475             0.7352    0.5938
HW3 criteria + phase LR     III     363 0.8266   0.7753          

In [11]:
# Compare false-success errors on the SAME unsuccessful validation trials and reason groups.
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["HW3 criteria + phase LR predicted success"] = processed_val_probability[failure_mask] >= 0.5
unsuccessful["HW3 combined + phase LR predicted success"] = combined_val_probability[failure_mask] >= 0.5

def compare_reason_errors(failures):
    rows = []
    model_names = ["Criteria + phase LR", "HW3 criteria + phase LR",
                   "HW3 combined + phase LR"]
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for name in model_names:
                errors = int(group[f"{name} predicted success"].sum())
                row[f"{name} count"] = errors
                row[f"{name} rate"] = errors / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

combined_reason_errors = compare_reason_errors(unsuccessful)
print(combined_reason_errors.round(3).to_string(index=False))

# A distinct candidate file; the existing raw-criteria submission remains unchanged.
combined_submission = pd.DataFrame({"ID": outcome_test.ID,
                                    "label": combined_test_probability})
assert combined_submission.ID.equals(sample_submission.ID)
assert combined_submission.ID.is_unique
assert combined_submission.label.notna().all()
assert np.isfinite(combined_submission.label).all()
assert combined_submission.label.between(0, 1).all()
combined_submission_path = OUTPUT_DIR / "part2_hw3_combined_submission.csv"
combined_submission.to_csv(combined_submission_path, index=False)
print("Saved separate candidate:", combined_submission_path)
print("Rows / probability range:", len(combined_submission),
      round(combined_submission.label.min(), 4), round(combined_submission.label.max(), 4))


Phase                Reason group  Trials  Observed  Predicted  Criteria + phase LR count  Criteria + phase LR rate  HW3 criteria + phase LR count  HW3 criteria + phase LR rate  HW3 combined + phase LR count  HW3 combined + phase LR rate
    I                      Safety       8         1          7                          4                     0.500                              4                         0.500                              4                         0.500
    I                     Funding       3         1          2                          2                     0.667                              1                         0.333                              1                         0.333
    I                    Efficacy       4         0          4                          3                     0.750                              4                         1.000                              4                         1.000
    I                 Recruitment      24       

In [12]:
# Controlled change: add years since 1970 to the existing combined-text + phase model.
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler

train_start_year = pd.to_numeric(outcome_train.start_year, errors="coerce")
val_start_year = pd.to_numeric(outcome_val.start_year, errors="coerce")
test_start_year = pd.to_numeric(outcome_test.start_year, errors="coerce")
train_year_median = float(train_start_year.median())

def year_feature(years, training_median):
    # Impute with the TRAIN median before taking the explicit 1970 offset.
    return (years.fillna(training_median).to_numpy(dtype=float) - 1970).reshape(-1, 1)

train_years_since_epoch = year_feature(train_start_year, train_year_median)
val_years_since_epoch = year_feature(val_start_year, train_year_median)
test_years_since_epoch = year_feature(test_start_year, train_year_median)
year_scaler = StandardScaler()
train_year_scaled = year_scaler.fit_transform(train_years_since_epoch)
val_year_scaled = year_scaler.transform(val_years_since_epoch)
test_year_scaled = year_scaler.transform(test_years_since_epoch)

# Existing combined_X matrices are untouched; only one train-scaled numeric column is appended.
combined_year_X_train = hstack([combined_X_train, csr_matrix(train_year_scaled)]).tocsr()
combined_year_X_val = hstack([combined_X_val, csr_matrix(val_year_scaled)]).tocsr()
combined_year_X_test = hstack([combined_X_test, csr_matrix(test_year_scaled)]).tocsr()
combined_year_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                         random_state=42)
combined_year_model.fit(combined_year_X_train, outcome_train.label.to_numpy(dtype=int))
combined_year_val_probability = combined_year_model.predict_proba(combined_year_X_val)[:, 1]
combined_year_test_probability = combined_year_model.predict_proba(combined_year_X_test)[:, 1]
combined_year_scores = evaluate_outcome_model(
    outcome_val, combined_year_val_probability, "HW3 combined + phase + year LR"
)
year_comparison_scores = pd.concat([
    validation_scores.loc[validation_scores.Model.eq("Criteria + phase LR")],
    combined_scores, combined_year_scores,
], ignore_index=True)
print(year_comparison_scores.round(4).to_string(index=False))
print("Train median start_year used for all missing values:", train_year_median)
print("Missing start_year train/val/test:",
      train_start_year.isna().sum(), val_start_year.isna().sum(), test_start_year.isna().sum())
print("Observed start_year train/val/test ranges:",
      [(int(s.min()), int(s.max())) for s in
       [train_start_year, val_start_year, test_start_year]])
print("Imputed years-since-1970 train/val/test ranges:",
      [(float(x.min()), float(x.max())) for x in
       [train_years_since_epoch, val_years_since_epoch, test_years_since_epoch]])
print("Scaled year train/val/test ranges:",
      [(round(float(x.min()), 3), round(float(x.max()), 3)) for x in
       [train_year_scaled, val_year_scaled, test_year_scaled]])
print("Feature matrix with year:", combined_year_X_train.shape)


                         Model   Phase  Trials     F1  ROC-AUC  Average precision  Log loss
           Criteria + phase LR       I     149 0.7692   0.7820             0.8380    0.5612
           Criteria + phase LR      II     501 0.6422   0.7358             0.7165    0.6013
           Criteria + phase LR     III     363 0.8300   0.7655             0.8578    0.5403
           Criteria + phase LR Overall    1013 0.7477   0.7689             0.8049    0.5736
       HW3 combined + phase LR       I     149 0.7471   0.7572             0.8124    0.5769
       HW3 combined + phase LR      II     501 0.6594   0.7493             0.7418    0.5904
       HW3 combined + phase LR     III     363 0.8266   0.7913             0.8850    0.5235
       HW3 combined + phase LR Overall    1013 0.7494   0.7779             0.8222    0.5644
HW3 combined + phase + year LR       I     149 0.7333   0.7588             0.8209    0.5750
HW3 combined + phase + year LR      II     501 0.6667   0.7476             0.741

In [13]:
# Error audit for raw criteria, combined text, and combined text plus start year.
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["HW3 combined + phase LR predicted success"] = combined_val_probability[failure_mask] >= 0.5
unsuccessful["HW3 combined + phase + year LR predicted success"] = combined_year_val_probability[failure_mask] >= 0.5

def year_reason_error_comparison(failures):
    names = ["Criteria + phase LR", "HW3 combined + phase LR",
             "HW3 combined + phase + year LR"]
    rows = []
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for name in names:
                mistakes = int(group[f"{name} predicted success"].sum())
                row[f"{name} false successes"] = mistakes
                row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

year_reason_errors = year_reason_error_comparison(unsuccessful)
print(year_reason_errors.round(3).to_string(index=False))

# Keep this candidate distinct from both existing submissions.
combined_year_submission = pd.DataFrame({"ID": outcome_test.ID,
                                         "label": combined_year_test_probability})
assert combined_year_submission.ID.equals(sample_submission.ID)
assert combined_year_submission.ID.is_unique
assert combined_year_submission.label.notna().all()
assert np.isfinite(combined_year_submission.label).all()
assert combined_year_submission.label.between(0, 1).all()
combined_year_submission_path = OUTPUT_DIR / "part2_hw3_combined_with_year_submission.csv"
combined_year_submission.to_csv(combined_year_submission_path, index=False)
print("Saved separate candidate:", combined_year_submission_path)
print("Rows / probability range:", len(combined_year_submission),
      round(combined_year_submission.label.min(), 4),
      round(combined_year_submission.label.max(), 4))


Phase                Reason group  Trials  Observed  Predicted  Criteria + phase LR false successes  Criteria + phase LR rate  HW3 combined + phase LR false successes  HW3 combined + phase LR rate  HW3 combined + phase + year LR false successes  HW3 combined + phase + year LR rate
    I                      Safety       8         1          7                                    4                     0.500                                        4                         0.500                                               4                                0.500
    I                     Funding       3         1          2                                    2                     0.667                                        1                         0.333                                               2                                0.667
    I                    Efficacy       4         0          4                                    3                     0.750                             

In [14]:
# Return to the no-year combined text model; compare L2 strengths only.
# Smaller C means stronger L2 regularization. C=1 reuses the existing fitted model.
REGULARIZATION_C_VALUES = [0.1, 0.3, 1.0]
regularized_models = {1.0: combined_model}
regularized_val_probabilities = {1.0: combined_val_probability}
regularized_test_probabilities = {1.0: combined_test_probability}
regularization_scores = []
for C in REGULARIZATION_C_VALUES:
    if C != 1.0:
        model = LogisticRegression(C=C, solver="liblinear",
                                   max_iter=1000, random_state=42)
        model.fit(combined_X_train, outcome_train.label.to_numpy(dtype=int))
        regularized_models[C] = model
        regularized_val_probabilities[C] = model.predict_proba(combined_X_val)[:, 1]
        regularized_test_probabilities[C] = model.predict_proba(combined_X_test)[:, 1]
    score = evaluate_outcome_model(outcome_val, regularized_val_probabilities[C],
                                   f"HW3 combined, no year, C={C:g}")
    score.insert(1, "C", C)
    regularization_scores.append(score)

regularization_scores = pd.concat(regularization_scores, ignore_index=True)
regularization_overall = regularization_scores.loc[
    regularization_scores.Phase.eq("Overall")
].sort_values("C")
best_c = float(regularization_overall.loc[
    regularization_overall["Log loss"].idxmin(), "C"
])
best_regularized_model = regularized_models[best_c]
best_regularized_val_probability = regularized_val_probabilities[best_c]
best_regularized_test_probability = regularized_test_probabilities[best_c]
print(regularization_scores.round(5).to_string(index=False))
print("Lowest validation log loss uses C=", best_c)
print("No-year feature matrix:", combined_X_train.shape)
print("The year-feature variant above is historical/rejected; it is not used here.")


                       Model   C   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
HW3 combined, no year, C=0.1 0.1       I     149 0.71503  0.71176            0.78955   0.61864
HW3 combined, no year, C=0.1 0.1      II     501 0.63319  0.72330            0.71411   0.62380
HW3 combined, no year, C=0.1 0.1     III     363 0.80416  0.77368            0.88180   0.57215
HW3 combined, no year, C=0.1 0.1 Overall    1013 0.72638  0.74412            0.79908   0.60454
HW3 combined, no year, C=0.3 0.3       I     149 0.73626  0.73787            0.80496   0.59563
HW3 combined, no year, C=0.3 0.3      II     501 0.64159  0.73778            0.73009   0.60511
HW3 combined, no year, C=0.3 0.3     III     363 0.80995  0.78554            0.88720   0.54635
HW3 combined, no year, C=0.3 0.3 Overall    1013 0.73517  0.76357            0.81577   0.58266
  HW3 combined, no year, C=1 1.0       I     149 0.74713  0.75717            0.81240   0.57688
  HW3 combined, no year, C=1 1.0      II     501 0

In [15]:
# Compare reason-group false successes for raw criteria, combined C=1, and best C.
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["HW3 combined C=1 predicted success"] = combined_val_probability[failure_mask] >= 0.5
unsuccessful["HW3 combined best C predicted success"] = best_regularized_val_probability[failure_mask] >= 0.5

def regularization_reason_errors(failures):
    model_names = ["Criteria + phase LR", "HW3 combined C=1", "HW3 combined best C"]
    rows = []
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for name in model_names:
                errors = int(group[f"{name} predicted success"].sum())
                row[f"{name} false successes"] = errors
                row[f"{name} rate"] = errors / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

regularization_reason_audit = regularization_reason_errors(unsuccessful)
print(regularization_reason_audit.round(3).to_string(index=False))

regularized_submission_path = None
if best_c != 1.0:
    regularized_submission = pd.DataFrame({"ID": outcome_test.ID,
                                           "label": best_regularized_test_probability})
    assert regularized_submission.ID.equals(sample_submission.ID)
    assert regularized_submission.ID.is_unique
    assert regularized_submission.label.notna().all()
    assert np.isfinite(regularized_submission.label).all()
    assert regularized_submission.label.between(0, 1).all()
    regularized_submission_path = OUTPUT_DIR / "part2_hw3_combined_regularized_submission.csv"
    regularized_submission.to_csv(regularized_submission_path, index=False)
    print("Saved distinct no-year candidate:", regularized_submission_path)
    print("Rows / probability range:", len(regularized_submission),
          round(regularized_submission.label.min(), 4),
          round(regularized_submission.label.max(), 4))
else:
    print("C=1 retained; existing no-year combined submission stays selected.")


Phase                Reason group  Trials  Observed  Predicted  Criteria + phase LR false successes  Criteria + phase LR rate  HW3 combined C=1 false successes  HW3 combined C=1 rate  HW3 combined best C false successes  HW3 combined best C rate
    I                      Safety       8         1          7                                    4                     0.500                                 4                  0.500                                    4                     0.500
    I                     Funding       3         1          2                                    2                     0.667                                 1                  0.333                                    1                     0.333
    I                    Efficacy       4         0          4                                    3                     0.750                                 4                  1.000                                    4                     1.000
    I           

In [16]:
# Bernoulli Naive Bayes uses presence/absence of each TF-IDF token and phase bit.
# Same no-year combined text and phase features as LR C=1; only classifier changes.
from sklearn.naive_bayes import BernoulliNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold

train_outcomes = outcome_train.label.to_numpy(dtype=int)
nb_model = BernoulliNB(alpha=1.0, binarize=0.0)
nb_model.fit(combined_X_train, train_outcomes)
nb_val_probability = nb_model.predict_proba(combined_X_val)[:, 1]
nb_test_probability = nb_model.predict_proba(combined_X_test)[:, 1]

# Calibration sees only training labels. A trial's multiple phase rows stay in one fold.
splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
grouped_folds = list(splitter.split(combined_X_train, train_outcomes,
                                    groups=outcome_train.trial_id.to_numpy()))
for fit_rows, calibration_rows in grouped_folds:
    assert set(outcome_train.trial_id.iloc[fit_rows]).isdisjoint(
        outcome_train.trial_id.iloc[calibration_rows])
    assert len(np.unique(train_outcomes[fit_rows])) == 2
    assert len(np.unique(train_outcomes[calibration_rows])) == 2
calibrated_nb_model = CalibratedClassifierCV(
    estimator=BernoulliNB(alpha=1.0, binarize=0.0), method="sigmoid",
    cv=grouped_folds, ensemble=False,
)
calibrated_nb_model.fit(combined_X_train, train_outcomes)
calibrated_nb_val_probability = calibrated_nb_model.predict_proba(combined_X_val)[:, 1]
calibrated_nb_test_probability = calibrated_nb_model.predict_proba(combined_X_test)[:, 1]

nb_scores = pd.concat([
    combined_scores,
    evaluate_outcome_model(outcome_val, nb_val_probability, "Bernoulli NB"),
    evaluate_outcome_model(outcome_val, calibrated_nb_val_probability,
                           "Bernoulli NB + train-only sigmoid"),
], ignore_index=True)
print(nb_scores.round(5).to_string(index=False))
print("Training-only calibration fold sizes:",
      [(len(fit_rows), len(calibration_rows)) for fit_rows, calibration_rows in grouped_folds])
print("Validation probability ranges LR / raw NB / calibrated NB:",
      [(round(float(p.min()), 5), round(float(p.max()), 5)) for p in
       [combined_val_probability, nb_val_probability, calibrated_nb_val_probability]])
print("Near-certain validation predictions (<.01 or >.99) raw / calibrated NB:",
      [int(((p < .01) | (p > .99)).sum()) for p in
       [nb_val_probability, calibrated_nb_val_probability]])


                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
          HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
          HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
          HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
          HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
                     Bernoulli NB       I     149 0.63694  0.66673            0.69932  10.40895
                     Bernoulli NB      II     501 0.65339  0.68541            0.61940  10.00451
                     Bernoulli NB     III     363 0.79113  0.66719            0.75736   8.85595
                     Bernoulli NB Overall    1013 0.71333  0.69154            0.69458   9.65242
Bernoulli NB + train-only sigmoid       I     149 0.63694  0.66397            0.68795   0.65824
Bernoulli NB + train-only sigmoid      I

In [17]:
# Inspect false-success trade-offs for each reason group on the same validation failures.
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["HW3 combined + phase LR predicted success"] = combined_val_probability[failure_mask] >= 0.5
unsuccessful["Bernoulli NB predicted success"] = nb_val_probability[failure_mask] >= 0.5
unsuccessful["Calibrated Bernoulli NB predicted success"] = calibrated_nb_val_probability[failure_mask] >= 0.5

def nb_reason_error_table(failures):
    name_to_column = {
        "Combined LR": "HW3 combined + phase LR predicted success",
        "Bernoulli NB": "Bernoulli NB predicted success",
        "Calibrated NB": "Calibrated Bernoulli NB predicted success",
    }
    rows = []
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for name, column in name_to_column.items():
                false_successes = int(group[column].sum())
                row[f"{name} false successes"] = false_successes
                row[f"{name} rate"] = false_successes / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

nb_reason_errors = nb_reason_error_table(unsuccessful)
print(nb_reason_errors.round(3).to_string(index=False))

nb_overall = nb_scores.loc[nb_scores.Phase.eq("Overall")]
best_nb_name = nb_overall.loc[
    nb_overall.Model.ne("HW3 combined + phase LR")
].sort_values("Log loss").iloc[0].Model
best_nb_test_probability = (calibrated_nb_test_probability
                            if best_nb_name == "Bernoulli NB + train-only sigmoid"
                            else nb_test_probability)
nb_submission = pd.DataFrame({"ID": outcome_test.ID,
                              "label": best_nb_test_probability})
assert nb_submission.ID.equals(sample_submission.ID)
assert nb_submission.ID.is_unique
assert nb_submission.label.notna().all()
assert np.isfinite(nb_submission.label).all()
assert nb_submission.label.between(0, 1).all()
nb_submission_path = OUTPUT_DIR / "part2_bernoulli_nb_experiment_submission.csv"
nb_submission.to_csv(nb_submission_path, index=False)
print("Best NB variant by validation log loss:", best_nb_name)
print("Saved experimental NB candidate:", nb_submission_path)
print("Rows / probability range:", len(nb_submission),
      round(nb_submission.label.min(), 5), round(nb_submission.label.max(), 5))


Phase                Reason group  Trials  Observed  Predicted  Combined LR false successes  Combined LR rate  Bernoulli NB false successes  Bernoulli NB rate  Calibrated NB false successes  Calibrated NB rate
    I                      Safety       8         1          7                            4             0.500                             3              0.375                              3               0.375
    I                     Funding       3         1          2                            1             0.333                             1              0.333                              1               0.333
    I                    Efficacy       4         0          4                            4             1.000                             4              1.000                              4               1.000
    I                 Recruitment      24         2         22                            5             0.208                             6              0.250  

In [18]:
# Retry HW3's binary unigram representation with Bernoulli NB.
# Keep the same cleaned four-field text and phase; fit vocabulary on train only.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold

binary_vectorizer = CountVectorizer(binary=True, min_df=5, ngram_range=(1, 1))
binary_train_text = binary_vectorizer.fit_transform(combined_train_text)
binary_val_text = binary_vectorizer.transform(combined_val_text)
binary_test_text = binary_vectorizer.transform(combined_test_text)
binary_X_train = hstack([binary_train_text,
                         phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
binary_X_val = hstack([binary_val_text,
                       phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
binary_X_test = hstack([binary_test_text,
                        phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
assert (binary_X_train.data >= 0).all()
assert np.isin(binary_X_train.data, [0, 1]).all()

train_outcomes = outcome_train.label.to_numpy(dtype=int)
grouped_folds = list(StratifiedGroupKFold(
    n_splits=3, shuffle=True, random_state=42
).split(binary_X_train, train_outcomes, groups=outcome_train.trial_id.to_numpy()))
for train_rows, calibration_rows in grouped_folds:
    assert set(outcome_train.trial_id.iloc[train_rows]).isdisjoint(
        outcome_train.trial_id.iloc[calibration_rows])
    assert len(np.unique(train_outcomes[train_rows])) == 2
    assert len(np.unique(train_outcomes[calibration_rows])) == 2

binary_nb = BernoulliNB(alpha=1.0, binarize=None)
binary_nb.fit(binary_X_train, train_outcomes)
binary_nb_val_probability = binary_nb.predict_proba(binary_X_val)[:, 1]
binary_nb_test_probability = binary_nb.predict_proba(binary_X_test)[:, 1]
binary_nb_calibrated = CalibratedClassifierCV(
    estimator=BernoulliNB(alpha=1.0, binarize=None), method="sigmoid",
    cv=grouped_folds, ensemble=False,
)
binary_nb_calibrated.fit(binary_X_train, train_outcomes)
binary_nb_cal_val_probability = binary_nb_calibrated.predict_proba(binary_X_val)[:, 1]
binary_nb_cal_test_probability = binary_nb_calibrated.predict_proba(binary_X_test)[:, 1]

count_nb_scores = pd.concat([
    combined_scores,
    evaluate_outcome_model(outcome_val, binary_nb_val_probability,
                           "HW3 binary-count Bernoulli NB"),
    evaluate_outcome_model(outcome_val, binary_nb_cal_val_probability,
                           "HW3 binary-count Bernoulli NB calibrated"),
], ignore_index=True)
print(count_nb_scores.round(5).to_string(index=False))
print("Binary vocabulary / total features:", len(binary_vectorizer.vocabulary_),
      binary_X_train.shape[1])
print("Near-certain validation probabilities raw / calibrated:",
      [int(((p < .01) | (p > .99)).sum()) for p in
       [binary_nb_val_probability, binary_nb_cal_val_probability]])
print("Train-only grouped calibration folds:",
      [(len(a), len(b)) for a, b in grouped_folds])


                                   Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
                 HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
                 HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
                 HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
                 HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
           HW3 binary-count Bernoulli NB       I     149 0.60927  0.67206            0.74371   8.42521
           HW3 binary-count Bernoulli NB      II     501 0.65306  0.70110            0.67562   7.49642
           HW3 binary-count Bernoulli NB     III     363 0.78558  0.73015            0.83257   6.24407
           HW3 binary-count Bernoulli NB Overall    1013 0.70719  0.71969            0.75859   7.18427
HW3 binary-count Bernoulli NB calibrated       I     149 0.60526  0.67105

In [19]:
# One further NB family on the SAME binary counts, since Bernoulli still trails LR.
multinomial_nb = MultinomialNB(alpha=1.0)
multinomial_nb.fit(binary_X_train, train_outcomes)
multinomial_nb_val_probability = multinomial_nb.predict_proba(binary_X_val)[:, 1]
multinomial_nb_test_probability = multinomial_nb.predict_proba(binary_X_test)[:, 1]
multinomial_nb_calibrated = CalibratedClassifierCV(
    estimator=MultinomialNB(alpha=1.0), method="sigmoid",
    cv=grouped_folds, ensemble=False,
)
multinomial_nb_calibrated.fit(binary_X_train, train_outcomes)
multinomial_nb_cal_val_probability = multinomial_nb_calibrated.predict_proba(binary_X_val)[:, 1]
multinomial_nb_cal_test_probability = multinomial_nb_calibrated.predict_proba(binary_X_test)[:, 1]
count_nb_scores = pd.concat([
    count_nb_scores,
    evaluate_outcome_model(outcome_val, multinomial_nb_val_probability,
                           "HW3 binary-count Multinomial NB"),
    evaluate_outcome_model(outcome_val, multinomial_nb_cal_val_probability,
                           "HW3 binary-count Multinomial NB calibrated"),
], ignore_index=True)
print(count_nb_scores.round(5).to_string(index=False))
print("Multinomial near-certain probabilities raw / calibrated:",
      [int(((p < .01) | (p > .99)).sum()) for p in
       [multinomial_nb_val_probability, multinomial_nb_cal_val_probability]])


                                     Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
                   HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
                   HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
                   HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
                   HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
             HW3 binary-count Bernoulli NB       I     149 0.60927  0.67206            0.74371   8.42521
             HW3 binary-count Bernoulli NB      II     501 0.65306  0.70110            0.67562   7.49642
             HW3 binary-count Bernoulli NB     III     363 0.78558  0.73015            0.83257   6.24407
             HW3 binary-count Bernoulli NB Overall    1013 0.70719  0.71969            0.75859   7.18427
  HW3 binary-count Bernoulli NB calibrated       I     

In [20]:
# Compare error groups, and save a separate best-NB candidate only if informative.
retry_probabilities = {
    "Bernoulli raw": (binary_nb_val_probability, binary_nb_test_probability),
    "Bernoulli calibrated": (binary_nb_cal_val_probability, binary_nb_cal_test_probability),
    "Multinomial raw": (multinomial_nb_val_probability, multinomial_nb_test_probability),
    "Multinomial calibrated": (multinomial_nb_cal_val_probability,
                               multinomial_nb_cal_test_probability),
}
retry_model_names = {
    "Bernoulli raw": "HW3 binary-count Bernoulli NB",
    "Bernoulli calibrated": "HW3 binary-count Bernoulli NB calibrated",
    "Multinomial raw": "HW3 binary-count Multinomial NB",
    "Multinomial calibrated": "HW3 binary-count Multinomial NB calibrated",
}
overall_retry = count_nb_scores.loc[
    count_nb_scores.Phase.eq("Overall") &
    count_nb_scores.Model.ne("HW3 combined + phase LR")
]
best_retry_name = next(name for name, full_name in retry_model_names.items()
                       if full_name == overall_retry.sort_values("Log loss").iloc[0].Model)
best_retry_val_probability, best_retry_test_probability = retry_probabilities[best_retry_name]
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["HW3 combined LR predicted success"] = combined_val_probability[failure_mask] >= 0.5
unsuccessful["Best count NB predicted success"] = best_retry_val_probability[failure_mask] >= 0.5

def count_nb_reason_errors(failures):
    rows = []
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for model_name, column in [
                ("Combined LR", "HW3 combined LR predicted success"),
                ("Best count NB", "Best count NB predicted success"),
            ]:
                count = int(group[column].sum())
                row[f"{model_name} false successes"] = count
                row[f"{model_name} rate"] = count / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

count_nb_reason_audit = count_nb_reason_errors(unsuccessful)
print("Best binary-count NB by validation log loss:", best_retry_name)
print(count_nb_reason_audit.round(3).to_string(index=False))
retry_submission = pd.DataFrame({"ID": outcome_test.ID,
                                 "label": best_retry_test_probability})
assert retry_submission.ID.equals(sample_submission.ID)
assert retry_submission.ID.is_unique and retry_submission.label.notna().all()
assert np.isfinite(retry_submission.label).all()
assert retry_submission.label.between(0, 1).all()
retry_submission_path = OUTPUT_DIR / "part2_hw3_binary_count_nb_experiment_submission.csv"
retry_submission.to_csv(retry_submission_path, index=False)
print("Separate experimental CSV:", retry_submission_path)
print("Rows / probability range:", len(retry_submission),
      round(retry_submission.label.min(), 5), round(retry_submission.label.max(), 5))


Best binary-count NB by validation log loss: Multinomial calibrated
Phase                Reason group  Trials  Observed  Predicted  Combined LR false successes  Combined LR rate  Best count NB false successes  Best count NB rate
    I                      Safety       8         1          7                            4             0.500                              3               0.375
    I                     Funding       3         1          2                            1             0.333                              1               0.333
    I                    Efficacy       4         0          4                            4             1.000                              3               0.750
    I                 Recruitment      24         2         22                            5             0.208                              5               0.208
    I                       Other       3         0          3                            1             0.333                  

In [21]:
# Pooled sigmoid calibration with preprocessing fitted separately in each training fold.
# Combined text cleaning is deterministic; learned TF-IDF/phase vocabulary is inside Pipeline.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.calibration import CalibratedClassifierCV

def calibration_inputs(text, df):
    return pd.DataFrame({"combined_text": pd.Series(text).reset_index(drop=True),
                         "phase": df.phase.reset_index(drop=True)})

cal_train = calibration_inputs(combined_train_text, outcome_train)
cal_val = calibration_inputs(combined_val_text, outcome_val)
cal_test = calibration_inputs(combined_test_text, outcome_test)
calibration_base_pipeline = Pipeline([
    ("features", ColumnTransformer([
        ("text", TfidfVectorizer(min_df=3, max_features=15000,
                                  ngram_range=(1, 2), sublinear_tf=True), "combined_text"),
        ("phase", OneHotEncoder(handle_unknown="ignore"), ["phase"]),
    ])),
    ("classifier", LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                       random_state=42)),
])
train_outcomes = outcome_train.label.to_numpy(dtype=int)
calibration_base_pipeline.fit(cal_train, train_outcomes)
pipeline_val_probability = calibration_base_pipeline.predict_proba(cal_val)[:, 1]
pipeline_test_probability = calibration_base_pipeline.predict_proba(cal_test)[:, 1]
max_val_difference = float(np.max(np.abs(pipeline_val_probability - combined_val_probability)))
max_test_difference = float(np.max(np.abs(pipeline_test_probability - combined_test_probability)))
assert max_val_difference < 1e-8 and max_test_difference < 1e-8

calibration_folds = list(StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
).split(cal_train, train_outcomes, groups=outcome_train.trial_id.to_numpy()))
for fit_rows, calibrate_rows in calibration_folds:
    assert set(outcome_train.trial_id.iloc[fit_rows]).isdisjoint(
        outcome_train.trial_id.iloc[calibrate_rows])
    assert len(np.unique(train_outcomes[fit_rows])) == 2
    assert len(np.unique(train_outcomes[calibrate_rows])) == 2
calibrated_lr = CalibratedClassifierCV(
    estimator=calibration_base_pipeline, method="sigmoid",
    cv=calibration_folds, ensemble=False,
)
calibrated_lr.fit(cal_train, train_outcomes)
calibrated_val_probability = calibrated_lr.predict_proba(cal_val)[:, 1]
calibrated_test_probability = calibrated_lr.predict_proba(cal_test)[:, 1]
calibrated_scores = evaluate_outcome_model(
    outcome_val, calibrated_val_probability, "HW3 combined LR + pooled sigmoid"
)
calibration_comparison = pd.concat([combined_scores, calibrated_scores],
                                   ignore_index=True)
print(calibration_comparison.round(5).to_string(index=False))
print("Full-fit pipeline max probability difference vs earlier LR, validation/test:",
      max_val_difference, max_test_difference)
print("Training-only trial-grouped fold sizes:",
      [(len(a), len(b)) for a, b in calibration_folds])
print("Probability ranges raw / calibrated validation:",
      [(round(float(p.min()), 5), round(float(p.max()), 5)) for p in
       [combined_val_probability, calibrated_val_probability]])


                           Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
         HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
         HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
         HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
         HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
HW3 combined LR + pooled sigmoid       I     149 0.75145  0.75717            0.81240   0.57520
HW3 combined LR + pooled sigmoid      II     501 0.65789  0.74926            0.74176   0.58887
HW3 combined LR + pooled sigmoid     III     363 0.82593  0.79129            0.88499   0.52037
HW3 combined LR + pooled sigmoid Overall    1013 0.74936  0.77785            0.82224   0.56231
Full-fit pipeline max probability difference vs earlier LR, validation/test: 0.0 0.0
Training-only trial-grouped fold sizes: [(6429, 1608), (6429

In [22]:
# Compare false-success errors without changing reason lookup or outcome features.
failure_mask = outcome_val.label.eq(0).to_numpy()
unsuccessful["Combined LR predicted success"] = combined_val_probability[failure_mask] >= 0.5
unsuccessful["Calibrated LR predicted success"] = calibrated_val_probability[failure_mask] >= 0.5

def calibration_reason_errors(failures):
    rows = []
    for phase in PHASES:
        phase_rows = failures.loc[failures.phase.eq(phase)]
        for reason in [*REASONS, "No recorded stopping reason"]:
            mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                    if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
            group = phase_rows.loc[mask]
            row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
                   "Observed": int(group.reason_source.eq("observed").sum()),
                   "Predicted": int(group.reason_source.eq("predicted").sum())}
            for name, column in [
                ("Raw combined LR", "Combined LR predicted success"),
                ("Calibrated LR", "Calibrated LR predicted success"),
            ]:
                mistakes = int(group[column].sum())
                row[f"{name} false successes"] = mistakes
                row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

calibration_reason_audit = calibration_reason_errors(unsuccessful)
print(calibration_reason_audit.round(3).to_string(index=False))

calibrated_submission = pd.DataFrame({"ID": outcome_test.ID,
                                      "label": calibrated_test_probability})
assert calibrated_submission.ID.equals(sample_submission.ID)
assert calibrated_submission.ID.is_unique and calibrated_submission.label.notna().all()
assert np.isfinite(calibrated_submission.label).all()
assert calibrated_submission.label.between(0, 1).all()
calibrated_submission_path = OUTPUT_DIR / "part2_hw3_combined_calibrated_submission.csv"
calibrated_submission.to_csv(calibrated_submission_path, index=False)
print("Saved distinct calibrated candidate:", calibrated_submission_path)
print("Rows / probability range:", len(calibrated_submission),
      round(calibrated_submission.label.min(), 5),
      round(calibrated_submission.label.max(), 5))


Phase                Reason group  Trials  Observed  Predicted  Raw combined LR false successes  Raw combined LR rate  Calibrated LR false successes  Calibrated LR rate
    I                      Safety       8         1          7                                4                 0.500                              4               0.500
    I                     Funding       3         1          2                                1                 0.333                              1               0.333
    I                    Efficacy       4         0          4                                4                 1.000                              4               1.000
    I                 Recruitment      24         2         22                                5                 0.208                              5               0.208
    I                       Other       3         0          3                                1                 0.333                              1       

In [23]:
# Controlled criteria change: retain selected negation and clinical-context words.
# Every other criteria-cleaning step and all other field cleaners match HW3 exactly.
meaningful_criteria_words = {
    "no", "not", "nor", "never", "without", "neither", "absent", "negative",
    "excluding", "excluded", "prior", "previous", "history", "treatment",
    "therapy", "disease", "age", "status", "within", "before", "after",
    "only", "must", "allowed",
}

def pre_process_criteria_keep_meaningful(value):
    if not isinstance(value, str) or not value.strip():
        return ""
    text = value.strip().lower().replace("inclusion criteria:", "").replace("exclusion criteria:", "")
    words = []
    for original_token in re.split(r"[\s\W]+", text):
        token = lemmatizer.lemmatize(original_token)
        if (token and (original_token in meaningful_criteria_words
                       or token in meaningful_criteria_words
                       or token not in criteria_stopwords)
                and not re.search(r"\d|®", token)):
            words.append(token)
    return " ".join(words)

def make_meaningful_combined_text(df):
    cleaned = pd.DataFrame({
        "criteria_pp": df.criteria.map(pre_process_criteria_keep_meaningful),
        "diseases_cleaned_pp": df.diseases_cleaned.map(pre_process_diseases_cleaned),
        "drugs_pp": df.drugs.map(pre_process_drugs),
        "icdcodes_pp": df.icdcodes.map(pre_process_icd),
    })
    return cleaned.astype(str).agg(" ".join, axis=1).str.strip(), cleaned

meaningful_train_text, meaningful_train_fields = make_meaningful_combined_text(outcome_train)
meaningful_val_text, meaningful_val_fields = make_meaningful_combined_text(outcome_val)
meaningful_test_text, meaningful_test_fields = make_meaningful_combined_text(outcome_test)
for split, old in [(meaningful_train_fields, combined_train_fields),
                   (meaningful_val_fields, combined_val_fields),
                   (meaningful_test_fields, combined_test_fields)]:
    for field in ["diseases_cleaned_pp", "drugs_pp", "icdcodes_pp"]:
        assert split[field].equals(old[field]), f"Unexpected change to {field}"

meaningful_tfidf = TfidfVectorizer(min_df=3, max_features=15000,
                                    ngram_range=(1, 2), sublinear_tf=True)
meaningful_X_train = hstack([
    meaningful_tfidf.fit_transform(meaningful_train_text),
    phase_encoder.transform(outcome_train[["phase"]]),
]).tocsr()
meaningful_X_val = hstack([
    meaningful_tfidf.transform(meaningful_val_text),
    phase_encoder.transform(outcome_val[["phase"]]),
]).tocsr()
meaningful_X_test = hstack([
    meaningful_tfidf.transform(meaningful_test_text),
    phase_encoder.transform(outcome_test[["phase"]]),
]).tocsr()
meaningful_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                      random_state=42)
meaningful_model.fit(meaningful_X_train, outcome_train.label.to_numpy(dtype=int))
meaningful_val_probability = meaningful_model.predict_proba(meaningful_X_val)[:, 1]
meaningful_test_probability = meaningful_model.predict_proba(meaningful_X_test)[:, 1]
meaningful_scores = evaluate_outcome_model(
    outcome_val, meaningful_val_probability, "Meaningful criteria + other HW3 fields + phase LR"
)
meaningful_comparison = pd.concat([combined_scores, meaningful_scores], ignore_index=True)
print(meaningful_comparison.round(5).to_string(index=False))
for example in ["No prior treatment or therapy", "Exclusion criteria: not allowed without previous disease history"]:
    print("EXAMPLE", repr(example))
    print("  original cleaning:", repr(pre_process_criteria(example)))
    print("  keep-meaningful:", repr(pre_process_criteria_keep_meaningful(example)))
old_counts = combined_train_fields.criteria_pp.str.split().map(len)
new_counts = meaningful_train_fields.criteria_pp.str.split().map(len)
print("Criteria tokens per training row, original/variant:",
      round(old_counts.mean(), 2), round(new_counts.mean(), 2))
print("Training criteria rows changed:",
      int((combined_train_fields.criteria_pp != meaningful_train_fields.criteria_pp).sum()),
      "/", len(outcome_train))
print("TF-IDF vocabulary original/variant:",
      len(combined_tfidf.vocabulary_), len(meaningful_tfidf.vocabulary_))


                                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
                          HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
                          HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
                          HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
                          HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
Meaningful criteria + other HW3 fields + phase LR       I     149 0.75000  0.75478            0.81305   0.57798
Meaningful criteria + other HW3 fields + phase LR      II     501 0.64348  0.74709            0.73963   0.59135
Meaningful criteria + other HW3 fields + phase LR     III     363 0.81481  0.79299            0.88655   0.52226
Meaningful criteria + other HW3 fields + phase LR Overall    1013 0.73810  0.77728            0.82287   

In [24]:
# Reason labels are only for error analysis; they are never classifier features.
meaningful_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
meaningful_failures["Original combined LR false success"] = (
    combined_val_probability[failure_mask] >= 0.5)
meaningful_failures["Meaningful criteria LR false success"] = (
    meaningful_val_probability[failure_mask] >= 0.5)
meaningful_reason_rows = []
for phase in PHASES:
    phase_rows = meaningful_failures.loc[meaningful_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for label in ["Original combined LR", "Meaningful criteria LR"]:
            errors = int(group[f"{label} false success"].sum())
            row[f"{label} false successes"] = errors
            row[f"{label} rate"] = errors / len(group) if len(group) else np.nan
        meaningful_reason_rows.append(row)
meaningful_reason_audit = pd.DataFrame(meaningful_reason_rows)
print(meaningful_reason_audit.round(3).to_string(index=False))

original_loss = float(combined_scores.loc[combined_scores.Phase.eq("Overall"), "Log loss"].iloc[0])
variant_loss = float(meaningful_scores.loc[meaningful_scores.Phase.eq("Overall"), "Log loss"].iloc[0])
if variant_loss < original_loss:
    meaningful_submission = pd.DataFrame({"ID": outcome_test.ID,
                                          "label": meaningful_test_probability})
    assert meaningful_submission.ID.equals(sample_submission.ID)
    assert meaningful_submission.ID.is_unique and len(meaningful_submission) == 1500
    assert np.isfinite(meaningful_submission.label).all()
    assert meaningful_submission.label.between(0, 1).all()
    meaningful_submission_path = OUTPUT_DIR / "part2_hw3_meaningful_criteria_submission.csv"
    meaningful_submission.to_csv(meaningful_submission_path, index=False)
    print("Saved validation-improving candidate:", meaningful_submission_path)
else:
    meaningful_submission_path = None
    print("No CSV exported: variant did not improve overall validation log loss.")


Phase                Reason group  Trials  Observed  Predicted  Original combined LR false successes  Original combined LR rate  Meaningful criteria LR false successes  Meaningful criteria LR rate
    I                      Safety       8         1          7                                     4                      0.500                                       4                        0.500
    I                     Funding       3         1          2                                     1                      0.333                                       2                        0.667
    I                    Efficacy       4         0          4                                     4                      1.000                                       4                        1.000
    I                 Recruitment      24         2         22                                     5                      0.208                                       6                        0.250
    I          

In [25]:
# Controlled n-gram comparison: same four cleaned fields, phase, LR, and training split.
# The vectorizer vocabulary is fitted separately on training text for each range.
def fit_combined_ngram_variant(ngram_range, model_name):
    vectorizer = TfidfVectorizer(min_df=3, max_features=15000,
                                 ngram_range=ngram_range, sublinear_tf=True)
    train_text_features = vectorizer.fit_transform(combined_train_text)
    val_text_features = vectorizer.transform(combined_val_text)
    test_text_features = vectorizer.transform(combined_test_text)
    X_train = hstack([train_text_features,
                      phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
    X_val = hstack([val_text_features,
                    phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
    X_test = hstack([test_text_features,
                     phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
    model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                               random_state=42)
    model.fit(X_train, outcome_train.label.to_numpy(dtype=int))
    val_probability = model.predict_proba(X_val)[:, 1]
    test_probability = model.predict_proba(X_test)[:, 1]
    scores = evaluate_outcome_model(outcome_val, val_probability, model_name)
    return {"model": model, "vectorizer": vectorizer, "train_features": train_text_features,
            "val_probability": val_probability, "test_probability": test_probability,
            "scores": scores, "X_train": X_train}

unigrams_only = fit_combined_ngram_variant((1, 1), "HW3 combined LR, unigrams only")
bigrams_only = fit_combined_ngram_variant((2, 2), "HW3 combined LR, bigrams only")
ngram_comparison = pd.concat([combined_scores, unigrams_only["scores"],
                              bigrams_only["scores"]], ignore_index=True)
print(ngram_comparison.round(5).to_string(index=False))

ngram_feature_summary = pd.DataFrame([
    {"N-grams": "Unigrams + bigrams", "Vocabulary": len(combined_tfidf.vocabulary_),
     "Average text nonzeros/row": combined_train_features.getnnz(axis=1).mean(),
     "Text density": combined_train_features.nnz / np.prod(combined_train_features.shape)},
    {"N-grams": "Unigrams only", "Vocabulary": len(unigrams_only["vectorizer"].vocabulary_),
     "Average text nonzeros/row": unigrams_only["train_features"].getnnz(axis=1).mean(),
     "Text density": unigrams_only["train_features"].nnz / np.prod(unigrams_only["train_features"].shape)},
    {"N-grams": "Bigrams only", "Vocabulary": len(bigrams_only["vectorizer"].vocabulary_),
     "Average text nonzeros/row": bigrams_only["train_features"].getnnz(axis=1).mean(),
     "Text density": bigrams_only["train_features"].nnz / np.prod(bigrams_only["train_features"].shape)},
])
print(ngram_feature_summary.round(5).to_string(index=False))


                         Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
       HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
       HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
       HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
       HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
HW3 combined LR, unigrams only       I     149 0.75556  0.75570            0.81970   0.57771
HW3 combined LR, unigrams only      II     501 0.64176  0.74778            0.74366   0.58853
HW3 combined LR, unigrams only     III     363 0.82528  0.79024            0.88602   0.52404
HW3 combined LR, unigrams only Overall    1013 0.74339  0.77597            0.82400   0.56383
 HW3 combined LR, bigrams only       I     149 0.75556  0.73989            0.79304   0.59580
 HW3 combined LR, bigrams only      II     501 0.65823  0.74282       

In [26]:
# False-success audit uses reasons only after predictions; never as model inputs.
ngram_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
for name, probabilities in [
    ("Mixed", combined_val_probability),
    ("Unigrams", unigrams_only["val_probability"]),
    ("Bigrams", bigrams_only["val_probability"]),
]:
    ngram_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5

ngram_reason_rows = []
for phase in PHASES:
    phase_rows = ngram_failures.loc[ngram_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name in ["Mixed", "Unigrams", "Bigrams"]:
            errors = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = errors
            row[f"{name} rate"] = errors / len(group) if len(group) else np.nan
        ngram_reason_rows.append(row)
ngram_reason_audit = pd.DataFrame(ngram_reason_rows)
print(ngram_reason_audit.round(3).to_string(index=False))

mixed_loss = float(combined_scores.loc[combined_scores.Phase.eq("Overall"), "Log loss"].iloc[0])
bigram_loss = float(bigrams_only["scores"].loc[
    bigrams_only["scores"].Phase.eq("Overall"), "Log loss"].iloc[0])
if bigram_loss < mixed_loss:
    bigram_submission = pd.DataFrame({"ID": outcome_test.ID,
                                      "label": bigrams_only["test_probability"]})
    assert bigram_submission.ID.equals(sample_submission.ID)
    assert bigram_submission.ID.is_unique and len(bigram_submission) == 1500
    assert np.isfinite(bigram_submission.label).all()
    assert bigram_submission.label.between(0, 1).all()
    bigram_submission_path = OUTPUT_DIR / "part2_hw3_combined_bigrams_only_submission.csv"
    bigram_submission.to_csv(bigram_submission_path, index=False)
    print("Exported validation-improving bigram candidate:", bigram_submission_path)
else:
    bigram_submission_path = None
    print("No bigram-only CSV exported: overall validation log loss did not improve.")


Phase                Reason group  Trials  Observed  Predicted  Mixed false successes  Mixed rate  Unigrams false successes  Unigrams rate  Bigrams false successes  Bigrams rate
    I                      Safety       8         1          7                      4       0.500                         4          0.500                        3         0.375
    I                     Funding       3         1          2                      1       0.333                         2          0.667                        1         0.333
    I                    Efficacy       4         0          4                      4       1.000                         4          1.000                        4         1.000
    I                 Recruitment      24         2         22                      5       0.208                         8          0.333                        8         0.333
    I                       Other       3         0          3                      1       0.333             

In [27]:
# Mixed unigram+bigram feature-cap test; all other training choices stay fixed.
def fit_combined_feature_cap(max_features):
    vectorizer = TfidfVectorizer(min_df=3, max_features=max_features,
                                 ngram_range=(1, 2), sublinear_tf=True)
    train_text_features = vectorizer.fit_transform(combined_train_text)
    val_text_features = vectorizer.transform(combined_val_text)
    test_text_features = vectorizer.transform(combined_test_text)
    X_train = hstack([train_text_features,
                      phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
    X_val = hstack([val_text_features,
                    phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
    X_test = hstack([test_text_features,
                     phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
    model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                               random_state=42)
    model.fit(X_train, outcome_train.label.to_numpy(dtype=int))
    val_probability = model.predict_proba(X_val)[:, 1]
    test_probability = model.predict_proba(X_test)[:, 1]
    scores = evaluate_outcome_model(
        outcome_val, val_probability, f"HW3 combined LR, {max_features:,} features"
    )
    return {"model": model, "vectorizer": vectorizer,
            "train_text_features": train_text_features,
            "val_probability": val_probability,
            "test_probability": test_probability, "scores": scores}

cap_30000 = fit_combined_feature_cap(30000)
cap_50000 = fit_combined_feature_cap(50000)
feature_cap_comparison = pd.concat([combined_scores, cap_30000["scores"],
                                    cap_50000["scores"]], ignore_index=True)
print(feature_cap_comparison.round(5).to_string(index=False))
feature_cap_summary = pd.DataFrame([
    {"Max features": 15000, "Vocabulary": len(combined_tfidf.vocabulary_),
     "Average text nonzeros/row": combined_train_features.getnnz(axis=1).mean()},
    *[
        {"Max features": cap, "Vocabulary": len(candidate["vectorizer"].vocabulary_),
         "Average text nonzeros/row": candidate["train_text_features"].getnnz(axis=1).mean()}
        for cap, candidate in [(30000, cap_30000), (50000, cap_50000)]
    ],
])
print(feature_cap_summary.round(2).to_string(index=False))


                           Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
         HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
         HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
         HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
         HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
HW3 combined LR, 30,000 features       I     149 0.73563  0.75331            0.81261   0.58124
HW3 combined LR, 30,000 features      II     501 0.65495  0.75269            0.74616   0.58898
HW3 combined LR, 30,000 features     III     363 0.82505  0.79367            0.88794   0.52360
HW3 combined LR, 30,000 features Overall    1013 0.74573  0.77891            0.82519   0.56442
HW3 combined LR, 50,000 features       I     149 0.75281  0.74908            0.81184   0.58466
HW3 combined LR, 50,000 features      II     501 0

In [28]:
# Reason-group error audit remains separate from prediction features.
cap_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
for name, probabilities in [("15k", combined_val_probability),
                            ("30k", cap_30000["val_probability"]),
                            ("50k", cap_50000["val_probability"])]:
    cap_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5

feature_cap_reason_rows = []
for phase in PHASES:
    phase_rows = cap_failures.loc[cap_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name in ["15k", "30k", "50k"]:
            mistakes = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = mistakes
            row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
        feature_cap_reason_rows.append(row)
feature_cap_reason_audit = pd.DataFrame(feature_cap_reason_rows)
print(feature_cap_reason_audit.round(3).to_string(index=False))

baseline_loss = float(combined_scores.loc[combined_scores.Phase.eq("Overall"),
                                          "Log loss"].iloc[0])
cap_candidates = [(30000, cap_30000), (50000, cap_50000)]
best_cap, best_candidate = min(
    cap_candidates,
    key=lambda item: float(item[1]["scores"].loc[
        item[1]["scores"].Phase.eq("Overall"), "Log loss"].iloc[0]),
)
best_loss = float(best_candidate["scores"].loc[
    best_candidate["scores"].Phase.eq("Overall"), "Log loss"].iloc[0])
if best_loss < baseline_loss:
    cap_submission = pd.DataFrame({"ID": outcome_test.ID,
                                   "label": best_candidate["test_probability"]})
    assert cap_submission.ID.equals(sample_submission.ID)
    assert cap_submission.ID.is_unique and len(cap_submission) == 1500
    assert np.isfinite(cap_submission.label).all()
    assert cap_submission.label.between(0, 1).all()
    cap_submission_path = OUTPUT_DIR / f"part2_hw3_combined_{best_cap}features_submission.csv"
    cap_submission.to_csv(cap_submission_path, index=False)
    print("Exported best validation-improving cap candidate:", cap_submission_path)
else:
    cap_submission_path = None
    print("No larger-cap CSV exported: neither improves overall validation log loss.")


Phase                Reason group  Trials  Observed  Predicted  15k false successes  15k rate  30k false successes  30k rate  50k false successes  50k rate
    I                      Safety       8         1          7                    4     0.500                    4     0.500                    4     0.500
    I                     Funding       3         1          2                    1     0.333                    1     0.333                    1     0.333
    I                    Efficacy       4         0          4                    4     1.000                    4     1.000                    4     1.000
    I                 Recruitment      24         2         22                    5     0.208                    6     0.250                    6     0.250
    I                       Other       3         0          3                    1     0.333                    1     0.333                    1     0.333
    I No recorded stopping reason      28         0          0  

In [29]:
# Conservative cleanup of lemmatizer artifacts only: "has" -> "ha", "was" -> "wa".
# The source training criteria contain neither standalone "ha" nor "wa".
# Do not change clinical terms, negation, age/dose handling, codes, drugs or other settings.
lemma_artifacts = {"ha", "wa"}

def pre_process_criteria_without_lemma_artifacts(value):
    return " ".join(word for word in pre_process_criteria(value).split()
                    if word not in lemma_artifacts)

def make_artifact_cleaned_combined_text(df):
    cleaned = pd.DataFrame({
        "criteria_pp": df.criteria.map(pre_process_criteria_without_lemma_artifacts),
        "diseases_cleaned_pp": df.diseases_cleaned.map(pre_process_diseases_cleaned),
        "drugs_pp": df.drugs.map(pre_process_drugs),
        "icdcodes_pp": df.icdcodes.map(pre_process_icd),
    })
    return cleaned.astype(str).agg(" ".join, axis=1).str.strip(), cleaned

artifact_train_text, artifact_train_fields = make_artifact_cleaned_combined_text(outcome_train)
artifact_val_text, artifact_val_fields = make_artifact_cleaned_combined_text(outcome_val)
artifact_test_text, artifact_test_fields = make_artifact_cleaned_combined_text(outcome_test)
for new_fields, old_fields in [(artifact_train_fields, combined_train_fields),
                               (artifact_val_fields, combined_val_fields),
                               (artifact_test_fields, combined_test_fields)]:
    for field in ["diseases_cleaned_pp", "drugs_pp", "icdcodes_pp"]:
        assert new_fields[field].equals(old_fields[field])

artifact_tfidf = TfidfVectorizer(min_df=3, max_features=15000,
                                 ngram_range=(1, 2), sublinear_tf=True)
artifact_train_features = artifact_tfidf.fit_transform(artifact_train_text)
artifact_val_features = artifact_tfidf.transform(artifact_val_text)
artifact_test_features = artifact_tfidf.transform(artifact_test_text)
artifact_X_train = hstack([artifact_train_features,
                           phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
artifact_X_val = hstack([artifact_val_features,
                         phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
artifact_X_test = hstack([artifact_test_features,
                          phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
artifact_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                    random_state=42)
artifact_model.fit(artifact_X_train, outcome_train.label.to_numpy(dtype=int))
artifact_val_probability = artifact_model.predict_proba(artifact_X_val)[:, 1]
artifact_test_probability = artifact_model.predict_proba(artifact_X_test)[:, 1]
artifact_scores = evaluate_outcome_model(
    outcome_val, artifact_val_probability, "HW3 combined LR without ha/wa artifacts"
)
artifact_comparison = pd.concat([combined_scores, artifact_scores], ignore_index=True)
print(artifact_comparison.round(5).to_string(index=False))

raw_criteria = outcome_train.criteria.fillna("").str.lower()
print("Raw training criteria with standalone has/was/ha/wa:",
      {token: int(raw_criteria.str.contains(r"\b" + token + r"\b", regex=True).sum())
       for token in ["has", "was", "ha", "wa"]})
old_criteria_words = combined_train_fields.criteria_pp.str.split()
new_criteria_words = artifact_train_fields.criteria_pp.str.split()
print("Artifact tokens in cleaned training criteria:",
      {token: int(old_criteria_words.map(lambda words: words.count(token)).sum())
       for token in sorted(lemma_artifacts)})
print("Rows changed:", int((combined_train_fields.criteria_pp != artifact_train_fields.criteria_pp).sum()),
      "/", len(outcome_train))
print("Average criteria tokens original/cleaned:",
      round(old_criteria_words.map(len).mean(), 2), round(new_criteria_words.map(len).mean(), 2))
print("Vocabulary original/cleaned:",
      len(combined_tfidf.vocabulary_), len(artifact_tfidf.vocabulary_))
print("Average active text features/row original/cleaned:",
      round(combined_train_features.getnnz(axis=1).mean(), 2),
      round(artifact_train_features.getnnz(axis=1).mean(), 2))
print("Example original/cleaned:",
      repr(pre_process_criteria("The patient has diabetes and was treated with insulin.")),
      repr(pre_process_criteria_without_lemma_artifacts(
          "The patient has diabetes and was treated with insulin.")))


                                  Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
                HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
                HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
                HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
                HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
HW3 combined LR without ha/wa artifacts       I     149 0.75145  0.75699            0.81190   0.57758
HW3 combined LR without ha/wa artifacts      II     501 0.65652  0.74960            0.74242   0.59031
HW3 combined LR without ha/wa artifacts     III     363 0.82657  0.79207            0.88611   0.52295
HW3 combined LR without ha/wa artifacts Overall    1013 0.74894  0.77798            0.82276   0.56430
Raw training criteria with standalone has/was/ha/wa: {'has': 2319, 'was': 687, 'ha

In [30]:
# Source-aware false-success audit; reason labels are never model inputs.
artifact_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
for name, probabilities in [("Original", combined_val_probability),
                            ("Cleaned", artifact_val_probability)]:
    artifact_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5

artifact_reason_rows = []
for phase in PHASES:
    phase_rows = artifact_failures.loc[artifact_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name in ["Original", "Cleaned"]:
            mistakes = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = mistakes
            row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
        artifact_reason_rows.append(row)
artifact_reason_audit = pd.DataFrame(artifact_reason_rows)
print(artifact_reason_audit.round(3).to_string(index=False))

baseline_loss = float(combined_scores.loc[combined_scores.Phase.eq("Overall"),
                                          "Log loss"].iloc[0])
artifact_loss = float(artifact_scores.loc[artifact_scores.Phase.eq("Overall"),
                                          "Log loss"].iloc[0])
# Require a substantive validation gain; a few ten-thousandths can be split noise.
meaningful_gain = baseline_loss - artifact_loss >= 0.001
if meaningful_gain:
    artifact_submission = pd.DataFrame({"ID": outcome_test.ID,
                                        "label": artifact_test_probability})
    assert artifact_submission.ID.equals(sample_submission.ID)
    assert artifact_submission.ID.is_unique and len(artifact_submission) == 1500
    assert np.isfinite(artifact_submission.label).all()
    assert artifact_submission.label.between(0, 1).all()
    artifact_submission_path = OUTPUT_DIR / "part2_hw3_without_lemma_artifacts_submission.csv"
    artifact_submission.to_csv(artifact_submission_path, index=False)
    print("Exported candidate with >=0.001 validation log-loss improvement:",
          artifact_submission_path)
else:
    artifact_submission_path = None
    print("No CSV exported: validation log-loss gain did not reach 0.001.")


Phase                Reason group  Trials  Observed  Predicted  Original false successes  Original rate  Cleaned false successes  Cleaned rate
    I                      Safety       8         1          7                         4          0.500                        4         0.500
    I                     Funding       3         1          2                         1          0.333                        1         0.333
    I                    Efficacy       4         0          4                         4          1.000                        4         1.000
    I                 Recruitment      24         2         22                         5          0.208                        5         0.208
    I                       Other       3         0          3                         1          0.333                        1         0.333
    I No recorded stopping reason      28         0          0                        12          0.429                       11         0.393

In [31]:
# Controlled minimum-document-frequency test: keep terms seen in at least five train rows.
# Text cleaning, mixed n-grams, 15k cap, phase encoder and LR C=1 remain fixed.
min_df5_tfidf = TfidfVectorizer(min_df=5, max_features=15000,
                                ngram_range=(1, 2), sublinear_tf=True)
min_df5_train_features = min_df5_tfidf.fit_transform(combined_train_text)
min_df5_val_features = min_df5_tfidf.transform(combined_val_text)
min_df5_test_features = min_df5_tfidf.transform(combined_test_text)
min_df5_X_train = hstack([min_df5_train_features,
                          phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
min_df5_X_val = hstack([min_df5_val_features,
                        phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
min_df5_X_test = hstack([min_df5_test_features,
                         phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
min_df5_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                                   random_state=42)
min_df5_model.fit(min_df5_X_train, outcome_train.label.to_numpy(dtype=int))
min_df5_val_probability = min_df5_model.predict_proba(min_df5_X_val)[:, 1]
min_df5_test_probability = min_df5_model.predict_proba(min_df5_X_test)[:, 1]
min_df5_scores = evaluate_outcome_model(outcome_val, min_df5_val_probability,
                                        "HW3 combined + phase LR, min_df=5")
min_df5_comparison = pd.concat([combined_scores, min_df5_scores], ignore_index=True)
print(min_df5_comparison.round(5).to_string(index=False))
print("TF-IDF vocabulary, min_df=3 / 5:",
      len(combined_tfidf.vocabulary_), len(min_df5_tfidf.vocabulary_))
print("Average active text features/row, min_df=3 / 5:",
      round(combined_train_features.getnnz(axis=1).mean(), 2),
      round(min_df5_train_features.getnnz(axis=1).mean(), 2))
print("Training matrix shapes, min_df=3 / 5:", combined_X_train.shape,
      min_df5_X_train.shape)


                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
          HW3 combined + phase LR       I     149 0.74713  0.75717            0.81240   0.57688
          HW3 combined + phase LR      II     501 0.65944  0.74926            0.74176   0.59040
          HW3 combined + phase LR     III     363 0.82657  0.79129            0.88499   0.52349
          HW3 combined + phase LR Overall    1013 0.74936  0.77785            0.82224   0.56443
HW3 combined + phase LR, min_df=5       I     149 0.74713  0.75699            0.81184   0.57689
HW3 combined + phase LR, min_df=5      II     501 0.65944  0.74912            0.74175   0.59048
HW3 combined + phase LR, min_df=5     III     363 0.82657  0.79143            0.88530   0.52335
HW3 combined + phase LR, min_df=5 Overall    1013 0.74936  0.77783            0.82242   0.56443
TF-IDF vocabulary, min_df=3 / 5: 15000 15000
Average active text features/row, min_df=3 / 5: 189.72 189.73
Training matrix shapes, min_d

In [32]:
# Source-aware error audit on unsuccessful validation rows; reasons are never model inputs.
min_df5_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
for name, probabilities in [("min_df=3", combined_val_probability),
                            ("min_df=5", min_df5_val_probability)]:
    min_df5_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5

min_df5_reason_rows = []
for phase in PHASES:
    phase_rows = min_df5_failures.loc[min_df5_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name in ["min_df=3", "min_df=5"]:
            mistakes = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = mistakes
            row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
        min_df5_reason_rows.append(row)
min_df5_reason_audit = pd.DataFrame(min_df5_reason_rows)
print(min_df5_reason_audit.round(3).to_string(index=False))

baseline_loss = float(combined_scores.loc[combined_scores.Phase.eq("Overall"),
                                          "Log loss"].iloc[0])
min_df5_loss = float(min_df5_scores.loc[min_df5_scores.Phase.eq("Overall"),
                                        "Log loss"].iloc[0])
if baseline_loss - min_df5_loss >= 0.001:
    min_df5_submission = pd.DataFrame({"ID": outcome_test.ID,
                                       "label": min_df5_test_probability})
    assert min_df5_submission.ID.equals(sample_submission.ID)
    assert min_df5_submission.ID.is_unique and len(min_df5_submission) == 1500
    assert np.isfinite(min_df5_submission.label).all()
    assert min_df5_submission.label.between(0, 1).all()
    min_df5_submission_path = OUTPUT_DIR / "part2_hw3_combined_min_df5_submission.csv"
    min_df5_submission.to_csv(min_df5_submission_path, index=False)
    print("Exported validation-improving candidate:", min_df5_submission_path)
else:
    min_df5_submission_path = None
    print("No CSV exported: overall validation log-loss gain was below 0.001.")


Phase                Reason group  Trials  Observed  Predicted  min_df=3 false successes  min_df=3 rate  min_df=5 false successes  min_df=5 rate
    I                      Safety       8         1          7                         4          0.500                         4          0.500
    I                     Funding       3         1          2                         1          0.333                         1          0.333
    I                    Efficacy       4         0          4                         4          1.000                         4          1.000
    I                 Recruitment      24         2         22                         5          0.208                         5          0.208
    I                       Other       3         0          3                         1          0.333                         1          0.333
    I No recorded stopping reason      28         0          0                        12          0.429                        12 

In [33]:
# Current personal candidate chosen by the user: min_df=5, mixed n-grams, 15k cap.
# Earlier min_df=3 models/submissions remain as historical comparisons.
current_part2_model = min_df5_model
current_part2_vectorizer = min_df5_tfidf
current_part2_validation_scores = min_df5_scores
current_part2_submission = pd.DataFrame({
    "ID": outcome_test.ID,
    "label": min_df5_test_probability,
})
assert current_part2_submission.ID.equals(sample_submission.ID)
assert current_part2_submission.ID.is_unique
assert len(current_part2_submission) == len(outcome_test) == 1500
assert np.isfinite(current_part2_submission.label).all()
assert current_part2_submission.label.between(0, 1).all()
current_part2_submission_path = OUTPUT_DIR / "part2_hw3_combined_min_df5_submission.csv"
current_part2_submission.to_csv(current_part2_submission_path, index=False)
print("Current user-selected candidate:", current_part2_submission_path)
print("Rows:", len(current_part2_submission), "Unique IDs:",
      current_part2_submission.ID.nunique())
print("Probability range:",
      round(current_part2_submission.label.min(), 5),
      round(current_part2_submission.label.max(), 5))
print("Overall validation metrics:")
print(current_part2_validation_scores.loc[
    current_part2_validation_scores.Phase.eq("Overall")].round(5).to_string(index=False))


Current user-selected candidate: [original local path] Machine Learning/midterm/applied-ml-midterm/original_attempts/part_2/part2_hw3_combined_min_df5_submission.csv
Rows: 1500 Unique IDs: 1500
Probability range: 0.12875 0.95553
Overall validation metrics:
                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
HW3 combined + phase LR, min_df=5 Overall    1013 0.74936  0.77783            0.82242   0.56443


In [34]:
# max_df comparison on current min_df=5 model; all other settings fixed.
def fit_min_df5_max_df(max_df):
    vectorizer = TfidfVectorizer(min_df=5, max_df=max_df, max_features=15000,
                                 ngram_range=(1, 2), sublinear_tf=True)
    train_text_features = vectorizer.fit_transform(combined_train_text)
    val_text_features = vectorizer.transform(combined_val_text)
    test_text_features = vectorizer.transform(combined_test_text)
    X_train = hstack([train_text_features,
                      phase_encoder.transform(outcome_train[["phase"]])]).tocsr()
    X_val = hstack([val_text_features,
                    phase_encoder.transform(outcome_val[["phase"]])]).tocsr()
    X_test = hstack([test_text_features,
                     phase_encoder.transform(outcome_test[["phase"]])]).tocsr()
    model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000,
                               random_state=42)
    model.fit(X_train, outcome_train.label.to_numpy(dtype=int))
    val_probability = model.predict_proba(X_val)[:, 1]
    test_probability = model.predict_proba(X_test)[:, 1]
    scores = evaluate_outcome_model(outcome_val, val_probability,
                                    f"HW3 combined min_df=5 max_df={max_df:g}")
    return {"model": model, "vectorizer": vectorizer,
            "train_text_features": train_text_features,
            "val_probability": val_probability,
            "test_probability": test_probability, "scores": scores}

max_df_09 = fit_min_df5_max_df(0.9)
reference_vocabulary = set(min_df5_tfidf.vocabulary_)
removed_09 = sorted(reference_vocabulary - set(max_df_09["vectorizer"].vocabulary_))
max_df_08 = None
removed_08 = None
if not removed_09:
    max_df_08 = fit_min_df5_max_df(0.8)
    removed_08 = sorted(reference_vocabulary - set(max_df_08["vectorizer"].vocabulary_))

max_df_scores = [min_df5_scores, max_df_09["scores"]]
if max_df_08 is not None:
    max_df_scores.append(max_df_08["scores"])
max_df_comparison = pd.concat(max_df_scores, ignore_index=True)
print(max_df_comparison.round(5).to_string(index=False))

reference_df = np.asarray((min_df5_train_features > 0).sum(axis=0)).ravel()
max_df_idx = int(reference_df.argmax())
most_common_term = min_df5_tfidf.get_feature_names_out()[max_df_idx]
print("Training rows:", len(outcome_train),
      "max_df=.9 cutoff:", 0.9 * len(outcome_train),
      "max_df=.8 cutoff:", 0.8 * len(outcome_train))
print("Most widespread selected term:", most_common_term,
      "in", int(reference_df.max()), "training rows")
print("Terms removed from current 15k vocabulary at .9:", len(removed_09), removed_09[:30])
if removed_08 is not None:
    print("Terms removed from current 15k vocabulary at .8:", len(removed_08), removed_08[:30])
print("Actual vocabulary / average active text features per training row:")
for name, vec, features in [
    ("default 1.0", min_df5_tfidf, min_df5_train_features),
    ("0.9", max_df_09["vectorizer"], max_df_09["train_text_features"]),
    *(([("0.8", max_df_08["vectorizer"], max_df_08["train_text_features"])]
       if max_df_08 is not None else [])),
]:
    print(name, len(vec.vocabulary_), round(features.getnnz(axis=1).mean(), 2))


                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
HW3 combined + phase LR, min_df=5       I     149 0.74713  0.75699            0.81184   0.57689
HW3 combined + phase LR, min_df=5      II     501 0.65944  0.74912            0.74175   0.59048
HW3 combined + phase LR, min_df=5     III     363 0.82657  0.79143            0.88530   0.52335
HW3 combined + phase LR, min_df=5 Overall    1013 0.74936  0.77783            0.82242   0.56443
 HW3 combined min_df=5 max_df=0.9       I     149 0.74713  0.75699            0.81184   0.57689
 HW3 combined min_df=5 max_df=0.9      II     501 0.65944  0.74912            0.74175   0.59048
 HW3 combined min_df=5 max_df=0.9     III     363 0.82657  0.79143            0.88530   0.52335
 HW3 combined min_df=5 max_df=0.9 Overall    1013 0.74936  0.77783            0.82242   0.56443
 HW3 combined min_df=5 max_df=0.8       I     149 0.74713  0.75699            0.81184   0.57689
 HW3 combined min_df=5 max_df=0.8      I

In [35]:
# Select .8 only if .9 filters zero terms and .8 filters some; otherwise keep requested .9.
if max_df_08 is not None and removed_08:
    current_max_df = 0.8
    current_max_df_candidate = max_df_08
else:
    current_max_df = 0.9
    current_max_df_candidate = max_df_09
current_part2_model = current_max_df_candidate["model"]
current_part2_vectorizer = current_max_df_candidate["vectorizer"]
current_part2_validation_scores = current_max_df_candidate["scores"]

# Reason-group audit after outcome prediction. Part 1 reasons never train the model.
max_df_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
audit_candidates = [("default 1.0", min_df5_val_probability),
                    ("0.9", max_df_09["val_probability"])]
if max_df_08 is not None:
    audit_candidates.append(("0.8", max_df_08["val_probability"]))
for name, probabilities in audit_candidates:
    max_df_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5
max_df_reason_rows = []
for phase in PHASES:
    phase_rows = max_df_failures.loc[max_df_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name, _ in audit_candidates:
            mistakes = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = mistakes
            row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
        max_df_reason_rows.append(row)
max_df_reason_audit = pd.DataFrame(max_df_reason_rows)
print(max_df_reason_audit.round(3).to_string(index=False))

current_max_df_submission = pd.DataFrame({
    "ID": outcome_test.ID,
    "label": current_max_df_candidate["test_probability"],
})
assert current_max_df_submission.ID.equals(sample_submission.ID)
assert current_max_df_submission.ID.is_unique and len(current_max_df_submission) == 1500
assert np.isfinite(current_max_df_submission.label).all()
assert current_max_df_submission.label.between(0, 1).all()
suffix = "08" if current_max_df == 0.8 else "09"
current_max_df_submission_path = OUTPUT_DIR / f"part2_hw3_combined_min_df5_max_df{suffix}_submission.csv"
current_max_df_submission.to_csv(current_max_df_submission_path, index=False)
print("Current user-selected max_df:", current_max_df)
print("Saved distinct selected candidate:", current_max_df_submission_path)
print("Rows / probability range:", len(current_max_df_submission),
      round(current_max_df_submission.label.min(), 5),
      round(current_max_df_submission.label.max(), 5))


Phase                Reason group  Trials  Observed  Predicted  default 1.0 false successes  default 1.0 rate  0.9 false successes  0.9 rate  0.8 false successes  0.8 rate
    I                      Safety       8         1          7                            4             0.500                    4     0.500                    4     0.500
    I                     Funding       3         1          2                            1             0.333                    1     0.333                    1     0.333
    I                    Efficacy       4         0          4                            4             1.000                    4     1.000                    4     1.000
    I                 Recruitment      24         2         22                            5             0.208                    5     0.208                    5     0.208
    I                       Other       3         0          3                            1             0.333                    1     0.333

In [36]:
# Effective max_df comparison: 0.5 versus default 1.0 and 0.9.
# Reuse fit_min_df5_max_df so the only changed setting is maximum document frequency.
max_df_05 = fit_min_df5_max_df(0.5)
absent_05 = sorted(reference_vocabulary - set(max_df_05["vectorizer"].vocabulary_))
cutoff_removed_05 = sorted(term for term in reference_vocabulary
                           if reference_df[min_df5_tfidf.vocabulary_[term]]
                           > 0.5 * len(outcome_train))
cap_displaced_05 = sorted(set(absent_05) - set(cutoff_removed_05))
removed_05 = cutoff_removed_05
max_df05_comparison = pd.concat([min_df5_scores, max_df_09["scores"],
                                  max_df_05["scores"]], ignore_index=True)
print(max_df05_comparison.round(5).to_string(index=False))
print("max_df=0.5 threshold:", 0.5 * len(outcome_train), "training rows")
print("Terms removed by max_df=0.5 cutoff:", len(cutoff_removed_05), cutoff_removed_05)
print("Other terms displaced by 15k feature cap:", len(cap_displaced_05), cap_displaced_05)
print("Cutoff-removed term document frequencies in baseline:",
      {term: int(reference_df[min_df5_tfidf.vocabulary_[term]])
       for term in removed_05})
print("Vocabulary and average active text features/row, baseline / 0.5:",
      len(min_df5_tfidf.vocabulary_), len(max_df_05["vectorizer"].vocabulary_),
      round(min_df5_train_features.getnnz(axis=1).mean(), 2),
      round(max_df_05["train_text_features"].getnnz(axis=1).mean(), 2))


                            Model   Phase  Trials      F1  ROC-AUC  Average precision  Log loss
HW3 combined + phase LR, min_df=5       I     149 0.74713  0.75699            0.81184   0.57689
HW3 combined + phase LR, min_df=5      II     501 0.65944  0.74912            0.74175   0.59048
HW3 combined + phase LR, min_df=5     III     363 0.82657  0.79143            0.88530   0.52335
HW3 combined + phase LR, min_df=5 Overall    1013 0.74936  0.77783            0.82242   0.56443
 HW3 combined min_df=5 max_df=0.9       I     149 0.74713  0.75699            0.81184   0.57689
 HW3 combined min_df=5 max_df=0.9      II     501 0.65944  0.74912            0.74175   0.59048
 HW3 combined min_df=5 max_df=0.9     III     363 0.82657  0.79143            0.88530   0.52335
 HW3 combined min_df=5 max_df=0.9 Overall    1013 0.74936  0.77783            0.82242   0.56443
 HW3 combined min_df=5 max_df=0.5       I     149 0.75145  0.75809            0.81184   0.57648
 HW3 combined min_df=5 max_df=0.5      I

In [37]:
# Source-aware false-success comparison. Reason labels are never predictive features.
max_df05_failures = unsuccessful.copy()
failure_mask = outcome_val.label.eq(0).to_numpy()
for name, probabilities in [("default 1.0", min_df5_val_probability),
                            ("0.9", max_df_09["val_probability"]),
                            ("0.5", max_df_05["val_probability"])]:
    max_df05_failures[f"{name} false success"] = probabilities[failure_mask] >= 0.5
max_df05_reason_rows = []
for phase in PHASES:
    phase_rows = max_df05_failures.loc[max_df05_failures.phase.eq(phase)]
    for reason in [*REASONS, "No recorded stopping reason"]:
        mask = (phase_rows.reason_source.eq("no recorded stopping reason")
                if reason == "No recorded stopping reason" else phase_rows[reason].eq(1))
        group = phase_rows.loc[mask]
        row = {"Phase": phase, "Reason group": reason, "Trials": len(group),
               "Observed": int(group.reason_source.eq("observed").sum()),
               "Predicted": int(group.reason_source.eq("predicted").sum())}
        for name in ["default 1.0", "0.9", "0.5"]:
            mistakes = int(group[f"{name} false success"].sum())
            row[f"{name} false successes"] = mistakes
            row[f"{name} rate"] = mistakes / len(group) if len(group) else np.nan
        max_df05_reason_rows.append(row)
max_df05_reason_audit = pd.DataFrame(max_df05_reason_rows)
print(max_df05_reason_audit.round(3).to_string(index=False))

loss_09 = float(max_df_09["scores"].loc[
    max_df_09["scores"].Phase.eq("Overall"), "Log loss"].iloc[0])
loss_05 = float(max_df_05["scores"].loc[
    max_df_05["scores"].Phase.eq("Overall"), "Log loss"].iloc[0])
if removed_05 and loss_05 < loss_09:
    current_max_df = 0.5
    current_max_df_candidate = max_df_05
    current_part2_model = max_df_05["model"]
    current_part2_vectorizer = max_df_05["vectorizer"]
    current_part2_validation_scores = max_df_05["scores"]
    max_df05_submission = pd.DataFrame({
        "ID": outcome_test.ID,
        "label": max_df_05["test_probability"],
    })
    assert max_df05_submission.ID.equals(sample_submission.ID)
    assert max_df05_submission.ID.is_unique and len(max_df05_submission) == 1500
    assert np.isfinite(max_df05_submission.label).all()
    assert max_df05_submission.label.between(0, 1).all()
    current_max_df_submission_path = OUTPUT_DIR / "part2_hw3_combined_min_df5_max_df05_submission.csv"
    max_df05_submission.to_csv(current_max_df_submission_path, index=False)
    print("Selected 0.5 as validation-improving filtered candidate:",
          current_max_df_submission_path)
else:
    current_max_df = 0.9
    current_max_df_candidate = max_df_09
    current_part2_model = max_df_09["model"]
    current_part2_vectorizer = max_df_09["vectorizer"]
    current_part2_validation_scores = max_df_09["scores"]
    current_max_df_submission_path = OUTPUT_DIR / "part2_hw3_combined_min_df5_max_df09_submission.csv"
    print("Kept 0.9 as current candidate; 0.5 did not improve validation log loss.")
print("Current candidate:", current_max_df, current_max_df_submission_path)


Phase                Reason group  Trials  Observed  Predicted  default 1.0 false successes  default 1.0 rate  0.9 false successes  0.9 rate  0.5 false successes  0.5 rate
    I                      Safety       8         1          7                            4             0.500                    4     0.500                    4     0.500
    I                     Funding       3         1          2                            1             0.333                    1     0.333                    1     0.333
    I                    Efficacy       4         0          4                            4             1.000                    4     1.000                    4     1.000
    I                 Recruitment      24         2         22                            5             0.208                    5     0.208                    5     0.208
    I                       Other       3         0          3                            1             0.333                    1     0.333